====================================================================
COSC 2060: Modern Data Engineering & AI Infrastructure
Unit 2 Synchronous Coding Clinic ("I Do, We Do, You Do")
Topic: The Anomaly Engine & Vectorized LazyFrames
====================================================================
Run each cell below one at a time (`Shift + Enter`) to inspect outputs step by step.
====================================================================


### PART 0: BASICS WARM-UP — POLARS SELECT, FILTER & WITH_COLUMNS


In [ ]:
print("\n>>> Executing PART 0: BASICS WARM-UP (Polars Expressions from Scratch)...")
import polars as pl

warmup_df = pl.DataFrame({
    "trans_id": [1, 2, 3],
    "gallons": [100.0, -5.0, 200.0],
    "amount_usd": [380.0, -19.0, 760.0]
})
# 1. Filter out negative sensor glitches with .filter(pl.col(...) > 0)
# 2. Add a basic calculated column with .with_columns()
warmup_clean = (
    warmup_df
    .filter(pl.col("gallons") > 0)
    .with_columns([
        (pl.col("amount_usd") / pl.col("gallons")).round(2).alias("price_per_gallon")
    ])
)
print("Part 0 Warm-Up — Basic Polars Filter & Calculated Column:")
print(warmup_clean)


### PART 1: 'I DO' — INSTRUCTOR DEMONSTRATION


In [ ]:
print("\n>>> Executing PART 1: 'I DO' — INSTRUCTOR DEMONSTRATION...")

# In-memory telemetry LazyFrame
clinic_data = pl.LazyFrame({
    "driver_id": [101, 101, 102, 102],
    "city": ["Cheyenne", "Denver", "Dallas", "Houston"],
    "gallons": [140.0, 420.0, None, 210.0],
    "fuel_type": ["Diesel", "Diesel", None, "DEF"],
    "odometer": [250000, 250100, 400000, 400240]
})

# Declarative expression pipeline:
# 1. Fill null gallons with median
# 2. Fill null fuel_type with 'UNKNOWN'
# 3. Add physical tank overfill flag (> 300 gallons)
cleaned_lazy = clinic_data.with_columns([
    pl.col("gallons").fill_null(pl.col("gallons").median()).alias("gallons_imputed"),
    pl.col("fuel_type").fill_null("UNKNOWN").alias("fuel_type_imputed"),
    (pl.col("gallons") > 300.0).alias("is_tank_overflow")
])

print("Polars Cleaned Plan Execution:")
print(cleaned_lazy.collect())


### PART 2: 'WE DO' — GUIDED CLASS PRACTICE


In [ ]:
print("\n>>> Executing PART 2: 'WE DO' — GUIDED CLASS PRACTICE...")
# Scale odometer strictly to [0.0, 1.0]
normalized_lazy = cleaned_lazy.with_columns([
    ((pl.col("odometer") - pl.col("odometer").min()) / 
     (pl.col("odometer").max() - pl.col("odometer").min())).alias("norm_odometer")
])

print("Normalized Telemetry Features:")
print(normalized_lazy.select(["driver_id", "odometer", "norm_odometer"]).collect())


### PART 3: 'YOU DO' — DRIVER 42 CLONED FUEL CARD WALKTHROUGH
Run this cell to inspect how `.shift(1).over('driver_id')` exposes Driver 42's cloned card row by row.


In [ ]:
from datetime import datetime
import polars as pl

driver42_df = pl.DataFrame({
    "driver_id": [42, 42],
    "timestamp": [datetime(2026, 3, 10, 8, 0), datetime(2026, 3, 10, 8, 45)],
    "mile_marker": [100.0, 1050.0],
    "gallons": [85.0, 92.0],
})

velocity_df = (
    driver42_df.sort(["driver_id", "timestamp"])
    .with_columns([
        pl.col("mile_marker").shift(1).over("driver_id").alias("prev_mile_marker"),
        pl.col("timestamp").shift(1).over("driver_id").alias("prev_timestamp"),
    ])
    .with_columns([
        (
            (pl.col("mile_marker") - pl.col("prev_mile_marker")).abs()
            / ((pl.col("timestamp") - pl.col("prev_timestamp")).dt.total_minutes() / 60.0)
        ).fill_null(0.0).alias("implied_speed_mph")
    ])
)
print(velocity_df)
